In [2]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-

# python check_syntax.py --tasks /mnt/data/public_150.json --inference /mnt/data/inference.json --show-preview

"""
StructEval Text (JSON/YAML/TOML/XML/CSV) syntax checker for inference outputs.

Inputs:
  - public_150.json     : task list (expects task_id, output_type, query, ...)
  - inference.json   : inference list (expects task_id, generation)

Outputs:
  - Overall parse success/failure stats
  - Per output_type stats
  - Detailed failure list per output_type: task_id + error kind + message

Notes:
  - This is a SYNTAX checker (parseability / well-formedness).
  - It tries to extract the most plausible payload from markdown code fences.
"""

from __future__ import annotations

import argparse
import csv
import json
import re
import sys
import textwrap
import xml.etree.ElementTree as ET
from collections import defaultdict, Counter
from dataclasses import dataclass
from typing import Any, Optional, Tuple, Dict, List

# YAML (PyYAML) is optional but recommended
try:
    import yaml  # type: ignore
    _HAS_YAML = True
except Exception:
    yaml = None
    _HAS_YAML = False

# TOML: Python 3.11+ has tomllib; fallback to tomli if installed.
try:
    import tomllib  # py3.11+
    _TOML_LOADS = tomllib.loads
except Exception:
    try:
        import tomli  # type: ignore
        _TOML_LOADS = tomli.loads
    except Exception:
        _TOML_LOADS = None


SUPPORTED = {"JSON", "YAML", "TOML", "XML", "CSV"}


@dataclass
class CheckResult:
    task_id: str
    output_type: str
    ok: bool
    error_kind: Optional[str] = None
    error_message: Optional[str] = None
    extracted_from: Optional[str] = None  # e.g. "fence:xml" / "fence:unknown" / "raw"
    payload_preview: Optional[str] = None


CODE_FENCE_RE = re.compile(
    r"```(?P<lang>[a-zA-Z0-9_\-+]*)\s*\n(?P<body>.*?)(?:\n```)",
    re.DOTALL
)

def normalize_output_type(s: str) -> str:
    s2 = (s or "").strip().upper()
    # tolerate common variants
    if s2 in ("YML",):
        return "YAML"
    return s2

def extract_payload(text: str, output_type: str) -> Tuple[str, str]:
    """
    Try to extract the actual structured text from the model generation.
    Preference:
      1) code fence with matching language label
      2) first code fence
      3) heuristics (XML: from <?xml or first '<'; others: raw trimmed)
    Returns: (payload, extracted_from)
    """
    if text is None:
        return "", "raw"

    t = text.strip()

    # Collect fences
    fences = []
    for m in CODE_FENCE_RE.finditer(t):
        lang = (m.group("lang") or "").strip().lower()
        body = (m.group("body") or "").strip()
        fences.append((lang, body))

    ot = output_type.strip().lower()

    # 1) Prefer fence that matches the type label
    if fences:
        # map output_type to typical fence labels
        preferred_langs = {
            "json": {"json"},
            "yaml": {"yaml", "yml"},
            "toml": {"toml"},
            "xml": {"xml", "html"},  # sometimes xml output is labeled html
            "csv": {"csv", "tsv"},
        }.get(ot, set())

        for lang, body in fences:
            if lang in preferred_langs:
                return body, f"fence:{lang or 'unknown'}"

        # 2) otherwise first fence
        lang0, body0 = fences[0]
        return body0, f"fence:{lang0 or 'unknown'}"

    # 3) No fences: heuristics
    if ot == "xml":
        # try from xml declaration
        idx = t.find("<?xml")
        if idx != -1:
            return t[idx:].strip(), "raw:from_xml_decl"
        # or from first tag
        idx2 = t.find("<")
        if idx2 != -1:
            return t[idx2:].strip(), "raw:from_first_tag"
        return t, "raw"
    else:
        # remove leading "Here is ..." lines if any, but keep it simple
        return t, "raw"


def classify_exception(output_type: str, e: Exception, payload: str) -> Tuple[str, str]:
    ot = output_type.upper()

    # JSON
    if ot == "JSON":
        if isinstance(e, json.JSONDecodeError):
            return "JSONDecodeError", str(e)
        return type(e).__name__, str(e)

    # YAML
    if ot == "YAML":
        if not _HAS_YAML:
            return "MissingDependency", "PyYAML not installed. `pip install pyyaml`"
        # PyYAML exception hierarchy varies; use names if available
        name = type(e).__name__
        # common: ScannerError, ParserError, ComposerError, ConstructorError, ReaderError
        if "ScannerError" in name:
            return "YAMLScannerError", str(e)
        if "ParserError" in name:
            return "YAMLParserError", str(e)
        if "ReaderError" in name:
            return "YAMLReaderError", str(e)
        return name, str(e)

    # TOML
    if ot == "TOML":
        if _TOML_LOADS is None:
            return "MissingDependency", "No tomllib (py3.11+) or tomli. Install: `pip install tomli`"
        name = type(e).__name__
        if "TOMLDecodeError" in name:
            return "TOMLDecodeError", str(e)
        return name, str(e)

    # XML
    if ot == "XML":
        if isinstance(e, ET.ParseError):
            return "XMLParseError", str(e)
        return type(e).__name__, str(e)

    # CSV
    if ot == "CSV":
        return type(e).__name__, str(e)

    return type(e).__name__, str(e)


def parse_json(payload: str) -> Any:
    return json.loads(payload)

def parse_yaml(payload: str) -> Any:
    if not _HAS_YAML:
        raise RuntimeError("PyYAML not installed. `pip install pyyaml`")
    return yaml.safe_load(payload)  # type: ignore

def parse_toml(payload: str) -> Any:
    if _TOML_LOADS is None:
        raise RuntimeError("No tomllib/tomli available.")
    return _TOML_LOADS(payload)  # type: ignore

def parse_xml(payload: str) -> Any:
    # ET.fromstring expects exactly one root element (well-formed)
    return ET.fromstring(payload)

def parse_csv_rectangular(payload: str) -> Any:
    """
    Parse CSV and enforce a rectangular table:
      - at least 1 row
      - all rows have the same number of columns
    """
    # Let csv module parse; handle embedded commas/quotes.
    reader = csv.reader(payload.splitlines())
    rows = list(reader)

    if len(rows) == 0:
        raise ValueError("CSVEmpty: no rows")

    # allow a single empty line? treat as empty
    if len(rows) == 1 and len(rows[0]) == 0:
        raise ValueError("CSVEmpty: single empty row")

    widths = [len(r) for r in rows]
    w0 = widths[0]
    # sometimes header is 1 col but later has more etc -> flag
    if any(w != w0 for w in widths):
        # include a compact diagnostic
        raise ValueError(f"CSVInconsistentColumns: widths={widths[:10]}{'...' if len(widths)>10 else ''}")

    return rows

def check_one(task_id: str, output_type: str, generation: str) -> CheckResult:
    ot = normalize_output_type(output_type)

    payload, extracted_from = extract_payload(generation or "", ot)
    preview = payload[:160].replace("\n", "\\n")

    try:
        if ot == "JSON":
            parse_json(payload)
        elif ot == "YAML":
            parse_yaml(payload)
        elif ot == "TOML":
            parse_toml(payload)
        elif ot == "XML":
            parse_xml(payload)
        elif ot == "CSV":
            parse_csv_rectangular(payload)
        else:
            return CheckResult(task_id, ot, False, "UnsupportedType", f"Unsupported output_type={ot}", extracted_from, preview)

        return CheckResult(task_id, ot, True, extracted_from=extracted_from, payload_preview=preview)

    except Exception as e:
        kind, msg = classify_exception(ot, e, payload)
        return CheckResult(task_id, ot, False, kind, msg, extracted_from, preview)


def load_tasks(path: str) -> Dict[str, Dict[str, Any]]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("tasks json must be a list")
    by_id: Dict[str, Dict[str, Any]] = {}
    for obj in data:
        if not isinstance(obj, dict):
            continue
        tid = obj.get("task_id")
        if isinstance(tid, str):
            by_id[tid] = obj
    return by_id

def load_inference(path: str) -> Dict[str, str]:
    with open(path, "r", encoding="utf-8") as f:
        data = json.load(f)
    if not isinstance(data, list):
        raise ValueError("inference json must be a list")
    by_id: Dict[str, str] = {}
    for obj in data:
        if not isinstance(obj, dict):
            continue
        tid = obj.get("task_id")
        gen = obj.get("generation")
        if isinstance(tid, str):
            by_id[tid] = gen if isinstance(gen, str) else ""
    return by_id

def print_summary(results: List[CheckResult]) -> None:
    total = len(results)
    ok = sum(1 for r in results if r.ok)
    fail = total - ok

    print(f"全体（{total}件）")
    print(f"パース成功: {ok} / {total} = {ok/total:.1%}")
    print(f"パース失敗: {fail}件")
    print()

    # per type stats
    by_type: Dict[str, List[CheckResult]] = defaultdict(list)
    for r in results:
        by_type[r.output_type].append(r)

    print("output_type 別")
    for ot in sorted(by_type.keys()):
        group = by_type[ot]
        gtotal = len(group)
        gok = sum(1 for r in group if r.ok)
        gfail = gtotal - gok
        print(f"{ot}: {gok}/{gtotal} = {gok/gtotal:.1%} (fail {gfail})")
    print()

def print_failures(results: List[CheckResult], show_preview: bool, max_per_type: int) -> None:
    by_type: Dict[str, List[CheckResult]] = defaultdict(list)
    for r in results:
        if not r.ok:
            by_type[r.output_type].append(r)

    if not by_type:
        print("失敗なし")
        return

    for ot in sorted(by_type.keys()):
        fails = by_type[ot]
        print("=" * 80)
        print(f"[{ot}] 失敗 {len(fails)}件")
        # group by error kind
        kinds = Counter([f.error_kind or "Unknown" for f in fails])
        kind_summary = ", ".join([f"{k}:{v}" for k, v in kinds.most_common()])
        print(f"内訳: {kind_summary}")
        print("-" * 80)

        for i, f in enumerate(fails[:max_per_type]):
            print(f"{i+1:02d}. task_id={f.task_id}")
            print(f"    kind={f.error_kind}")
            # indent message for readability
            msg = (f.error_message or "").strip()
            if msg:
                msg = msg.replace("\n", " | ")
                print(f"    message={msg}")
            if f.extracted_from:
                print(f"    extracted_from={f.extracted_from}")
            if show_preview and f.payload_preview is not None:
                print(f"    payload_preview={f.payload_preview}")
        if len(fails) > max_per_type:
            print(f"... (省略: {len(fails)-max_per_type}件)")
        print()

def run(tasks: str = "public_150.json",
        inference: str = "inference.json",
        show_preview: bool = False,
        max_per_type: int = 200) -> int:
    tasks_dict = load_tasks(tasks)
    inf_dict = load_inference(inference)

    results: List[CheckResult] = []
    missing_inference = 0
    missing_task = 0

    for tid, task in tasks_dict.items():
        ot = normalize_output_type(str(task.get("output_type", "")))
        if ot not in SUPPORTED:
            results.append(CheckResult(tid, ot, False, "UnsupportedType", f"output_type={ot} not supported"))
            continue

        if tid not in inf_dict:
            missing_inference += 1
            results.append(CheckResult(tid, ot, False, "MissingInference", "No generation found"))
            continue

        gen = inf_dict.get(tid, "")
        results.append(check_one(tid, ot, gen))

    for tid in inf_dict.keys():
        if tid not in tasks_dict:
            missing_task += 1

    if not _HAS_YAML:
        print("[WARN] PyYAML が見つかりません。YAMLの検査をするには `pip install pyyaml` を実行してください。", file=sys.stderr)
    if _TOML_LOADS is None:
        print("[WARN] tomllib/tomli が見つかりません。TOMLの検査には Python3.11+ または `pip install tomli` が必要です。", file=sys.stderr)

    print_summary(results)

    if missing_inference:
        print(f"[INFO] 推論結果が存在しない task_id: {missing_inference}件")
    if missing_task:
        print(f"[INFO] inference 側にのみ存在し task 側に無い task_id: {missing_task}件")
    if missing_inference or missing_task:
        print()

    print_failures(results, show_preview=show_preview, max_per_type=max_per_type)

    any_fail = any(not r.ok for r in results)
    return 1 if any_fail else 0


def main(argv: Optional[List[str]] = None) -> int:
    ap = argparse.ArgumentParser(
        description="Syntax checker for StructEval Text format conversion outputs."
    )
    ap.add_argument("--tasks", default="public_150.json", help="Path to task json (public_150.json)")
    ap.add_argument("--inference", default="inference.json", help="Path to inference json (inference.json)")
    ap.add_argument("--show-preview", action="store_true", help="Show payload preview for failures")
    ap.add_argument("--max-per-type", type=int, default=200, help="Max failures to print per type")
    args = ap.parse_args(argv)  # ← ここがポイント（argv を渡せる）

    return run(
        tasks=args.tasks,
        inference=args.inference,
        show_preview=args.show_preview,
        max_per_type=args.max_per_type,
    )


if __name__ == "__main__":
    # Jupyter では SystemExit させない（kernel を巻き込むため）
    if "ipykernel" in sys.modules:
        # main([])  # ipykernelの -f を無視
        pass
    else:
        raise SystemExit(main())




In [3]:
exit_code = run(
    tasks="./public_150.json",
    inference="./inference.json",
    show_preview=True,
    max_per_type=200,
)
exit_code

""" or
# Notebook では ipykernel の -f を無視したいので argv を明示
exit_code = main([
    "--tasks", "./public_150.json",
    "--inference", "./inference.json",
    "--show-preview",
])
exit_code
"""

全体（150件）
パース成功: 147 / 150 = 98.0%
パース失敗: 3件

output_type 別
CSV: 20/20 = 100.0% (fail 0)
JSON: 50/50 = 100.0% (fail 0)
TOML: 23/25 = 92.0% (fail 2)
XML: 19/20 = 95.0% (fail 1)
YAML: 35/35 = 100.0% (fail 0)

[TOML] 失敗 2件
内訳: TOMLDecodeError:2
--------------------------------------------------------------------------------
01. task_id=p_8548b0aabd75cf81e240777a
    kind=TOMLDecodeError
    message=Invalid initial character for a key part (at line 7, column 6)
    extracted_from=raw
    payload_preview=[museum]\nname = "The Art of Tomorrow"\nlocation = { city = "New York", country = "United States" }\nestablished_year = 1995\ndirector = { full_name = "Dr. Evelyn R
02. task_id=p_10bb379b2cc851f7cb253007
    kind=TOMLDecodeError
    message=Invalid initial character for a key part (at line 8, column 4)
    extracted_from=raw
    payload_preview=[museum]\nname = "The Lumina Art Gallery"\nlocation = { city = "San Francisco", country = "United States" }\nestablished_year = 1985\ndirector = "Dr.

' or\n# Notebook では ipykernel の -f を無視したいので argv を明示\nexit_code = main([\n    "--tasks", "./public_150.json",\n    "--inference", "./inference.json",\n    "--show-preview",\n])\nexit_code\n'